# Anti-spoof classifier training notebook

This notebook provides a lightweight PyTorch training scaffold for an anti-spoof classifier.

In [ ]:
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

DATA_ROOT = Path('../data/anti_spoof')
TRAIN_ROOT = DATA_ROOT / 'train'
VAL_ROOT = DATA_ROOT / 'val'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

class SimpleImageDataset(Dataset):
    def __init__(self, root: Path, transform=None):
        self.root = root
        self.transform = transform
        self.samples = []
        for label_dir in sorted(self.root.iterdir()):
            if not label_dir.is_dir():
                continue
            for image_path in label_dir.glob('*'):
                if image_path.is_file():
                    self.samples.append((image_path, int(label_dir.name)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label = self.samples[idx]
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

train_ds = SimpleImageDataset(TRAIN_ROOT, transform=transform)
val_ds = SimpleImageDataset(VAL_ROOT, transform=transform)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

class AntiSpoofClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 2),
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.classifier(x)

model = AntiSpoofClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
    print(f'epoch {epoch + 1} complete')

torch.save(model.state_dict(), '../artifacts/anti_spoof_classifier.pt')
print('Saved model to ../artifacts/anti_spoof_classifier.pt')